# OOF Stacking
This notebook takes the out-of-fold (OOF) predictions from the various base models and trains a meta-model (stacking) to improve the final prediction score.

# True Out-of-Fold (OOF) Stacking Pipeline with Full Optuna Tuning

In [ ]:
# This notebook implements a rigorous 2-layer stacking ensemble using LightGBM, XGBoost, and CatBoost.
# It includes full Optuna hyperparameter tuning for XGBoost and CatBoost to maximize performance.
# All base models are trained on `train_original.csv` to learn the native NaN signals.
# The meta-model is Logistic Regression.

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import optuna
import warnings

warnings.filterwarnings('ignore')

## 1. Load Data & Preprocessing

In [ ]:
print("Loading data...")
train = pd.read_csv('../datasets/train_original.csv')
test = pd.read_csv('../datasets/test.csv')

X = train.drop(['id', 'addicted_label'], axis=1)
y = train['addicted_label']
X_test = test.drop(['id'], axis=1)

# Proper Ordinal Mappings
stress_mapping = {'Low': 0, 'Medium': 1, 'High': 2, 'Unknown': -1}
impact_mapping = {'No': 0, 'Yes': 1, 'Unknown': -1}

def apply_mappings(df):
    df_out = df.copy()
    df_out['stress_level'] = df_out['stress_level'].map(stress_mapping).fillna(-1).astype(int)
    df_out['academic_work_impact'] = df_out['academic_work_impact'].map(impact_mapping).fillna(-1).astype(int)
    df_out['gender'] = df_out['gender'].fillna('Unknown').astype('category')
    return df_out

X_preprocessed = apply_mappings(X)
X_test_preprocessed = apply_mappings(X_test)

# Feature Engineering
def add_features(df):
    df_out = df.copy()
    denom_screen = df_out['daily_screen_time_hours'].replace(0, 0.001)
    denom_notif = df_out['notifications_per_day'].replace(0, 0.001)

    df_out['social_media_ratio'] = df_out['social_media_hours'] / denom_screen
    df_out['gaming_ratio'] = df_out['gaming_hours'] / denom_screen
    df_out['work_study_ratio'] = df_out['work_study_hours'] / denom_screen
    df_out['app_opens_per_hour'] = df_out['app_opens_per_day'] / denom_screen
    df_out['notifications_to_opens_ratio'] = df_out['app_opens_per_day'] / denom_notif
    df_out['sleep_deficit'] = 8.0 - df_out['sleep_hours']
    return df_out

X_preprocessed = add_features(X_preprocessed)
X_test_preprocessed = add_features(X_test_preprocessed)

categorical_cols = ['gender']
for col in categorical_cols:
    X_preprocessed[col] = X_preprocessed[col].astype('category')
    X_test_preprocessed[col] = X_test_preprocessed[col].astype('category')

# Constants
N_FOLDS = 5

## 2. Load Pre-tuned Parameters for XGBoost & CatBoost

In [ ]:
import json

try:
    with open('../datasets/tuned_parameters.json', 'r') as f:
        tuned_params = json.load(f)
    xgb_best_params = tuned_params['xgboost']
    cat_best_params = tuned_params['catboost']
    print("Successfully loaded pre-tuned parameters for XGBoost and CatBoost.")
except FileNotFoundError:
    print("Error: tuned_parameters.json not found. Please run the original Optuna tuning first.")
    raise

## 4. LightGBM Optimal Parameters (Already Tuned)

In [ ]:
lgbm_best_params = {
    'n_estimators': 909, 
    'learning_rate': 0.081595, 
    'num_leaves': 66, 
    'max_depth': 9, 
    'min_child_samples': 49, 
    'colsample_bytree': 0.526310, 
    'subsample': 0.778710, 
    'subsample_freq': 5, 
    'reg_alpha': 9.75536, 
    'reg_lambda': 0.001679, 
    'min_split_gain': 0.39105,
    'is_unbalance': True,
    'random_state': 42,
    'verbose': -1,
    'n_jobs': -1
}


# (Removed duplicate saving code)

## 5. Generate Out-of-Fold (OOF) Predictions

In [ ]:
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# Matrices to store OOF predictions for Layer 1
oof_train_lgb = np.zeros(len(X_preprocessed))
oof_train_xgb = np.zeros(len(X_preprocessed))
oof_train_cat = np.zeros(len(X_preprocessed))

# Matrices to store test predictions across folds
test_preds_lgb = np.zeros(len(X_test_preprocessed))
test_preds_xgb = np.zeros(len(X_test_preprocessed))
test_preds_cat = np.zeros(len(X_test_preprocessed))

print(f"Starting {N_FOLDS}-Fold CV OOF generation with best parameters...")

for fold, (train_idx, val_idx) in enumerate(cv.split(X_preprocessed, y)):
    print(f"--- Fold {fold + 1}/{N_FOLDS} ---")

    X_tr, y_tr = X_preprocessed.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X_preprocessed.iloc[val_idx], y.iloc[val_idx]

    # 1. LightGBM
    print("  Training LightGBM...")
    model_lgb = lgb.LGBMClassifier(**lgbm_best_params)
    model_lgb.fit(X_tr, y_tr)
    oof_train_lgb[val_idx] = model_lgb.predict_proba(X_val)[:, 1]
    test_preds_lgb += model_lgb.predict_proba(X_test_preprocessed)[:, 1] / N_FOLDS
    print(f"    LGBM Fold {fold + 1} AUC: {roc_auc_score(y_val, oof_train_lgb[val_idx]):.5f}")

    # 2. XGBoost
    print("  Training XGBoost...")
    model_xgb = xgb.XGBClassifier(**xgb_best_params)
    model_xgb.fit(X_tr, y_tr)
    oof_train_xgb[val_idx] = model_xgb.predict_proba(X_val)[:, 1]
    test_preds_xgb += model_xgb.predict_proba(X_test_preprocessed)[:, 1] / N_FOLDS
    print(f"    XGB Fold {fold + 1} AUC: {roc_auc_score(y_val, oof_train_xgb[val_idx]):.5f}")

    # 3. CatBoost
    print("  Training CatBoost...")
    model_cat = CatBoostClassifier(**cat_best_params)
    model_cat.fit(X_tr, y_tr)
    oof_train_cat[val_idx] = model_cat.predict_proba(X_val)[:, 1]
    test_preds_cat += model_cat.predict_proba(X_test_preprocessed)[:, 1] / N_FOLDS
    print(f"    CatBoost Fold {fold + 1} AUC: {roc_auc_score(y_val, oof_train_cat[val_idx]):.5f}")

## 6. Evaluate Base Models

In [ ]:
print("\nBase Models Final OOF ROC-AUC:")
print(f"LGBM OOF AUC:     {roc_auc_score(y, oof_train_lgb):.5f}")
print(f"XGBoost OOF AUC:  {roc_auc_score(y, oof_train_xgb):.5f}")
print(f"CatBoost OOF AUC: {roc_auc_score(y, oof_train_cat):.5f}")

## 7. Layer 2: Meta-Model Training (Stacking)

In [ ]:
X_train_meta = pd.DataFrame({'lgb': oof_train_lgb, 'xgb': oof_train_xgb, 'cat': oof_train_cat})
X_test_meta = pd.DataFrame({'lgb': test_preds_lgb, 'xgb': test_preds_xgb, 'cat': test_preds_cat})

try:
    oof_train_et = np.load('../datasets/oof_preds/oof_train_et.npy')
    test_preds_et = np.load('../datasets/oof_preds/test_preds_et.npy')
    
    X_train_meta['et'] = oof_train_et
    X_test_meta['et'] = test_preds_et
    print("Successfully loaded advanced diverse models OOF predictions.")
except FileNotFoundError:
    print("Advanced model OOF arrays not found. Make sure to run diverse_models_oof.py first. Proceeding with tree models only.")


print("\nTraining Meta-Model (Logistic Regression)...")
meta_model = LogisticRegression(random_state=42)
meta_model.fit(X_train_meta, y)

# Evaluate Meta-Model using CV
meta_oof_preds = np.zeros(len(X_train_meta))
for train_idx, val_idx in cv.split(X_train_meta, y):
    meta_cv_model = LogisticRegression(random_state=42)
    meta_cv_model.fit(X_train_meta.iloc[train_idx], y.iloc[train_idx])
    meta_oof_preds[val_idx] = meta_cv_model.predict_proba(X_train_meta.iloc[val_idx])[:, 1]

print(f"Final Stacking OOF AUC: {roc_auc_score(y, meta_oof_preds):.6f}")

print("\nMeta-Model Weights:")
for model_name, weight in zip(X_train_meta.columns, meta_model.coef_[0]):
    print(f"{model_name.upper():>8s}: {weight:.4f}")

## 8. Predict and Save Final Submission

In [ ]:
print("\nPredicting final test results...")
final_preds = meta_model.predict_proba(X_test_meta)[:, 1]

os.makedirs('../submissions', exist_ok=True)
submission = pd.DataFrame({'id': test['id'], 'addicted_label': final_preds})
submission.to_csv('../submissions/oof_stacking_optuna_submission.csv', index=False)
print("Submission saved to submissions/oof_stacking_optuna_submission.csv")